# 01 — Exploratory Data Analysis

Runs against the live DuckDB warehouse. Re-run after `make ingest dbt-build`.

## Outputs
1. Return distribution by year and by VIX tercile
2. Sector correlation heatmap
3. Realized vol vs VIX scatter
4. Yield curve over time
5. K-means of macro days + elbow plot
6. HMM 3-state pilot fit on VIX + 10Y + DXY changes, state probs over time

In [ ]:
from __future__ import annotations
import duckdb, polars as pl, plotly.express as px, plotly.io as pio
from pathlib import Path
from regime.config import get_settings

S = get_settings()
CHARTS = Path('../docs/charts'); CHARTS.mkdir(parents=True, exist_ok=True)
con = duckdb.connect(str(S.duckdb_path), read_only=True)

def save(fig, name):
    p = CHARTS / f'{name}.json'
    pio.write_json(fig, p)
    return p

In [ ]:
# 1. Return distribution by VIX tercile
df = con.execute('''
    select r.trade_date, r.log_ret_1d, m.vix
    from main_intermediate.int_returns r
    join main_marts.mart_macro_features m on m.feature_date = r.trade_date
    where r.ticker = 'SPY' and r.log_ret_1d is not null and m.vix is not null
''').pl()
df = df.with_columns(pl.col('vix').qcut(3, labels=['low','mid','high']).alias('vix_t'))
fig = px.histogram(df.to_pandas(), x='log_ret_1d', color='vix_t', nbins=80, barmode='overlay', opacity=0.6, title='SPY daily returns by VIX tercile')
save(fig, '01_returns_by_vix_tercile')
fig.show()

In [ ]:
# 2. Sector correlation heatmap
sectors = ['XLK','XLF','XLE','XLY','XLP','XLV','XLI','XLU','XLB','XLRE','XLC']
q = f"select ticker, trade_date, log_ret_1d from main_intermediate.int_returns where ticker in ({','.join(repr(t) for t in sectors)}) and log_ret_1d is not null"
rets = con.execute(q).pl().pivot(values='log_ret_1d', index='trade_date', on='ticker').drop_nulls()
corr = rets.drop('trade_date').to_pandas().corr()
fig = px.imshow(corr, text_auto='.2f', title='Sector ETF return correlation')
save(fig, '02_sector_correlation')
fig.show()